# SVA Noise Robustness — Colab Runner (LSTM)
Trains + evaluates the LSTM on all 5 noise conditions on a T4.

**Before running:** set `Runtime > Change runtime type > T4 GPU`.
Put the 6 data TSVs in a Drive folder and point `DATA_DIR` at it.

In [ ]:
# 1. Clone repo
!git clone https://github.com/blue-octopus235/FinalProject-LMs-CogModels-DSC291.git
%cd FinalProject-LMs-CogModels-DSC291
!pip install -q lemminflect

In [ ]:
# 2. Mount Drive and point at the data folder (the 6 TSVs)
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/dsc291_data'  # <-- EDIT to your folder
import os; print(sorted(os.listdir(DATA_DIR)))

In [ ]:
# 3. Config — KEEP max_sentences CONSTANT across conditions
MAX_SENTENCES = 150000   # ~minutes/epoch on T4; raise for the paper
EPOCHS = 5
SEEDS = [1]              # add [1,2,3] for the paper-phase seed runs
CONDITIONS = ['baseline','low','medium_low','medium_high','high']

In [ ]:
# 4. Train all conditions x seeds
import subprocess
for seed in SEEDS:
    for cond in CONDITIONS:
        print(f'==== train {cond} seed{seed} ====')
        subprocess.run(['python','src/train.py','--condition',cond,
            '--data_dir',DATA_DIR,'--max_sentences',str(MAX_SENTENCES),
            '--epochs',str(EPOCHS),'--seed',str(seed)], check=True)

In [ ]:
# 5. Evaluate all checkpoints
for seed in SEEDS:
    for cond in CONDITIONS:
        ckpt = f'checkpoints/lstm_{cond}_seed{seed}.pt'
        subprocess.run(['python','src/evaluate.py','--checkpoint',ckpt,
            '--condition',cond,'--data_dir',DATA_DIR,'--seed',str(seed)], check=True)

In [ ]:
# 6. Plots + table
subprocess.run(['python','src/plots.py'], check=True)
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('results/eval_results.csv'))
for p in ['acc_vs_noise.png','attractor_gap.png','acc_by_attractor.png']:
    display(Image(f'results/{p}'))

In [ ]:
# 7. (optional) copy results back to Drive so the team can see them
!cp -r results $DATA_DIR/../dsc291_results 2>/dev/null; print('done')